# Lab 5: Debugging Across the Dataflow Stack

**Course**: *Dataflow Architectures for AI: From Principles to Practice*
**Week**: 5, Debugging Across the Dataflow Stack
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)
**Runtime**: Google Colab, GPU runtime required (T4)
**Expected time**: 100-120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Localize a fault that the runtime never reports, with forward hooks and a first-divergence search against a clean baseline.
2. Detect and count a recompile storm with the recompile log, `TORCH_TRACE` and `tlparse`, remove it with shape buckets, and guard the fix.
3. Separate a precision difference from a logic bug with the FP64 criterion, then name the compiler layer with a four-backend ablation.
4. Turn a silent CPU fallback into an error, and find its host round trips on a profiler timeline.
5. Place each failure in the 8-layer model and say how it would surface on a dataflow target.

## Prerequisites
Labs 1-4 complete, plus the Week 5 lecture: the TRACE method, the 8-layer model, and Modules 3-5.

## How the lab maps to the lecture

| Part | Planted failure | TRACE step | Debugging layer | Lecture topics |
|---|---|---|---|---|
| 1 | Injected NaN in Qwen 2.5 0.5B | T, E | 1 (Python source) | Error at the next sync (Module 1), forward hooks and first divergence (Module 5) |
| 2 | Recompile storm | A, C | 2 (graph capture) | Compile logs and tlparse (Module 3), missed buckets and the compile guard (Module 6) |
| 3 | Silent divergence from a bad decomposition | A, E | 4 (decomposition) | Backend ablation and IR per stage (Module 3), FP64 reference (Module 4) |
| 4 | Silent CPU fallback in a dataflow backend | C, A | 4 (op mapping) | CPU fallback (Modules 2 and 4), profiler timeline (Module 3), strict-xfail tests (Module 6) |

## How this Self-Assessment Lab works

**Practice Lab**: You run this lab on **Google Colab** (free T4).
**OPTIONAL SELF GRADING:** **A separate grader notebook is available on Coursera Labs for self-evaluation**.

| | Where | What happens |
|---|---|---|
| **1. This notebook** | Google Colab (T4 GPU) | Complete every `TODO`, make every **Checkpoint** pass, then run the final cell to write **`lab5_results.json`** |
| **2. Grader notebook** | Coursera Labs (no GPU) | For automated self assessment, paste the contents of `lab5_results.json`, then `Run All`. **40 ungraded points** |

The grader notebook has no GPU and no network, so it cannot re-run anything.
It checks that your numbers are **internally consistent**: the first NaN cannot
come before the first non-finite value, a compile count must fit under the
recompile limit, a pass or fail must follow from your two RMSE values, and so
on. Those relations hold for any real run on any GPU, which is why a free-tier
T4 earns a full score.

> **Timings are compared with each other, never with a fixed value.** Colab VMs
> are shared. The grader checks that each derived number follows from *your* raw
> measurements.

| Part | Points | Comes from |
|---|---:|---|
| A: Structural checks | 12 | Parts 1-5 |
| B: Consistency checks | 20 | Parts 1-4 |
| C: Plausibility checks | 8 | Parts 2-4 |
| **Ungraded notebook total** | **40** | auto-graded from your JSON |

**Graded Quiz points:** a 10-point Concept Check quiz on Coursera, mandatory, 80% to pass.

### Working through this lab
1. Set **Runtime -> Change runtime type -> T4 GPU** before you run anything.
2. **Run the Preflight cell first, in a fresh runtime.** It sets `TORCH_TRACE`, which PyTorch reads only once, when `torch` is first imported.
3. Run each cell in order, fill in every `TODO`, and make every **Checkpoint** pass.
4. Run the final cell to produce **`lab5_results.json`**, then paste it into the grader notebook.

## Environment Preflight

This cell detects your runtime and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells run on CPU with the same code

It also sets `TORCH_TRACE` **before** `torch` is imported. PyTorch reads that
variable once, at import time (`torch/_logging/_internal.py`), so setting it
later in the notebook silently produces no trace.

In [ ]:
# === Environment Preflight ===
import os
import sys

# Part 2 reads compile IDs from a structured trace. TORCH_TRACE must be set
# before the first `import torch` in this runtime.
TRACE_DIR = "/content/lab5_torch_trace" if os.path.isdir("/content") else os.path.abspath("lab5_torch_trace")
os.makedirs(TRACE_DIR, exist_ok=True)
if "torch" in sys.modules and os.environ.get("TORCH_TRACE") != TRACE_DIR:
    print("[!] torch was already imported in this runtime, so TORCH_TRACE will not take effect.")
    print("    Runtime -> Restart session, then run this cell first. Part 2 can still run without it.")
os.environ["TORCH_TRACE"] = TRACE_DIR

assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# TESTED_TORCH_MIN and TESTED_TORCH are the oldest and newest torch versions this lab
# was run on top to bottom on a T4. Newer versions usually work, but the log text,
# compile IDs and kernel names that Parts 2 to 4 read can change between releases.
TESTED_TORCH_MIN = "2.4"
TESTED_TORCH = "2.15"
INSTALL_HINT = 'pip install "torch>=2.4,<2.16"'
try:
    import torch
except ImportError:
    raise RuntimeError(f"PyTorch not installed. Run: {INSTALL_HINT}")
_torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
assert _torch_version >= (2, 4), f"PyTorch >= 2.4 required, got {torch.__version__}. Run: {INSTALL_HINT}"
if _torch_version > tuple(int(x) for x in TESTED_TORCH.split('.')):
    print(f"WARNING: Tested on torch {TESTED_TORCH}. Log formats and kernel names may differ on newer versions.")
print(f"torch.__version__ = {torch.__version__}, torch.cuda.is_available() = {torch.cuda.is_available()}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"║  Trace dir: {TRACE_DIR[-37:]:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. The lab still runs on CPU, more slowly.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")

---
## 0: Setup

All installs happen here. `transformers` is pinned to the version every lab in
this course uses; `tlparse` turns the `TORCH_TRACE` log into an HTML report.
The cell then records every package version, so your results can be compared.

In [ ]:
!pip -q install transformers==4.44.2 tlparse

import contextlib
import copy
import glob
import json
import math
import shutil
import subprocess
import time
import warnings
from collections import OrderedDict
from importlib import metadata as _md

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch._dynamo as dynamo
from torch._dynamo.utils import counters
from torch.profiler import profile, ProfilerActivity
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt

torch.manual_seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
# One numeric path per run: no TF32 shortcuts on Ampere-class GPUs.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


def _ver(pkg):
    try:
        return _md.version(pkg)
    except Exception:
        return "not installed"


VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": _ver("transformers"),
    "tlparse": _ver("tlparse"),
    "device": torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU only",
}
for _k, _v in VERSIONS.items():
    print(f"{_k:13s}: {_v}")

# Course palette
TERRACOTTA, CREAM, DARK, SECONDARY, BORDER = "#B85C2C", "#FAF6F2", "#1A1A1A", "#5C5C5C", "#D4C4B8"


def sync():
    if DEV == "cuda":
        torch.cuda.synchronize()

In [ ]:
# PyTorch version compatibility check
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}.')
_limit_name = "recompile_limit" if hasattr(dynamo.config, "recompile_limit") else "cache_size_limit"
RECOMPILE_LIMIT = int(getattr(dynamo.config, _limit_name))
print(f'✓ PyTorch version compatible | dynamo.config.{_limit_name} = {RECOMPILE_LIMIT}')

---
## 1: ★ TODO: Find an Injected NaN

**Scenario.** A teammate swapped one activation in Qwen 2.5 0.5B for a "faster"
version. Generation now returns garbage. Nothing raised an exception.

This is the situation from Module 1: the runtime never tells you where the
fault is. You will use the Module 5 method instead. Record every module's
output with **forward hooks**, find the **first module whose output is not
finite**, and confirm it is the **first divergence** from a clean baseline.

TRACE steps: **T** (the symptom is "wrong values, no error") and **E**
(experiment against a baseline, one variable changed).

In [ ]:
# 1.0 (provided): load the course model in float32, eager attention.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
try:
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    qwen = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, attn_implementation="eager")
    MODEL_SOURCE = MODEL_ID
except Exception as e:
    # No network: a small, randomly initialised model of the same family.
    print(f"[!] Could not download {MODEL_ID} ({type(e).__name__}); using a small random Qwen2.")
    from transformers import Qwen2Config, Qwen2ForCausalLM
    _cfg = Qwen2Config(vocab_size=4096, hidden_size=256, intermediate_size=704,
                       num_hidden_layers=8, num_attention_heads=4, num_key_value_heads=2,
                       max_position_embeddings=512, attn_implementation="eager")
    qwen = Qwen2ForCausalLM(_cfg)
    tok = None
    MODEL_SOURCE = "random-qwen2-small"

qwen = qwen.to(DEV).eval()
N_LAYERS = qwen.config.num_hidden_layers
PROMPT = "Dataflow accelerators fix the schedule before the program runs, so"
if tok is not None:
    input_ids = tok(PROMPT, return_tensors="pt").input_ids.to(DEV)
else:
    input_ids = torch.randint(0, qwen.config.vocab_size, (1, 16), device=DEV)

with torch.no_grad():
    base_logits = qwen(input_ids, use_cache=False).logits
print(f"model: {MODEL_SOURCE} | layers: {N_LAYERS} | prompt tokens: {input_ids.shape[1]}")
print(f"baseline logits finite: {bool(torch.isfinite(base_logits).all())}")

In [ ]:
# TODO 1.1: Write a forward-hook recorder (lecture Module 5, forward hooks).
#
# record_outputs(model, inputs) must:
#   * register a forward hook on every named module except the root ("")
#   * in each hook, unwrap a tuple output (keep out[0]) and keep only tensors;
#     with transformers 4.44 the decoder layers return tuples
#   * store a detached float32 CPU copy in `records`, keyed by module path, in
#     the order the hooks FIRE (a module's hook fires when the module finishes)
#   * run ONE forward pass under torch.no_grad() with use_cache=False
#   * remove every hook before returning, even if the forward pass raises
#     (keep the handles; use try/finally)
# Return `records`.

def record_outputs(model, inputs):
    records = OrderedDict()
    handles = []
    # YOUR CODE HERE
    raise NotImplementedError
    return records

In [ ]:
# 1.2 (provided): record the clean baseline.
baseline = record_outputs(qwen, input_ids)
print(f"baseline: {len(baseline)} module outputs recorded")
print("first five:", list(baseline)[:5])

In [ ]:
# 1.3 (provided): FAULT INJECTION. Do not read this cell until Checkpoint 1 passes.
class FastGate(nn.Module):
    # A "faster" SiLU from a quantisation experiment. It is not.
    def forward(self, x):
        return x * torch.sigmoid(x) * (1 + 1e-3 * torch.log(x + x.abs()))


FAULT_LAYER = (3 * N_LAYERS) // 8
_orig_act = qwen.model.layers[FAULT_LAYER].mlp.act_fn
qwen.model.layers[FAULT_LAYER].mlp.act_fn = FastGate()

with torch.no_grad():
    faulty_logits = qwen(input_ids, use_cache=False).logits
print("The forward pass completed. No exception was raised.")
print(f"faulty logits finite: {bool(torch.isfinite(faulty_logits).all())} | "
      f"NaN fraction: {torch.isnan(faulty_logits).float().mean().item():.3f}")
faulty = record_outputs(qwen, input_ids)
print(f"faulty run: {len(faulty)} module outputs recorded")

In [ ]:
# TODO 1.4: Localize the fault from the two records.
#
# Walk the faulty record IN ORDER (list(faulty)) and find:
#   idx_nonfinite, first_nonfinite : first module whose output has any inf or NaN
#   idx_nan, first_nan             : first module whose output has any NaN
#   idx_divergent, first_divergent : first module that diverges from `baseline`,
#        i.e. its output is non-finite, or
#        max|faulty - baseline| > 1e-4 * (max|baseline| + 1e-6)
# Each idx_* is the position in list(faulty); each first_* is the module path.
# Also set baseline_nonfinite_count: how many baseline outputs are non-finite.
#
# Hint: torch.isfinite(t).all(), torch.isnan(t).any()

names = list(faulty)
# YOUR CODE HERE
# idx_nonfinite, first_nonfinite = ...
# idx_nan, first_nan = ...
# idx_divergent, first_divergent = ...
# baseline_nonfinite_count = ...
raise NotImplementedError

In [ ]:
print(f"first non-finite : [{idx_nonfinite}] {first_nonfinite}")
print(f"first NaN        : [{idx_nan}] {first_nan}")
print(f"first divergence : [{idx_divergent}] {first_divergent}")
if first_nonfinite in faulty:
    _t = faulty[first_nonfinite]
    print(f"\n{first_nonfinite}: +inf={int(torch.isposinf(_t).sum())}, "
          f"-inf={int(torch.isneginf(_t).sum())}, NaN={int(torch.isnan(_t).sum())} of {_t.numel()}")

In [ ]:
# TODO 1.5: Would torch.autograd.detect_anomaly have caught this?
#
# Run the faulty model once, forward only, inside
#     with torch.autograd.detect_anomaly(check_nan=True):
#         with torch.no_grad():
#             ...
# Set anomaly_raised = True if it raised an exception, False if it completed.
# (Lecture Module 3, tools on three stacks: detect_anomaly checks the BACKWARD pass.)

# YOUR CODE HERE
# anomaly_raised = ...
raise NotImplementedError

In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
assert len(faulty) >= 50, f"expected a record of every module, got {len(faulty)} entries"
assert baseline_nonfinite_count == 0, "the clean baseline should be finite everywhere"
assert first_nonfinite is not None, "the faulty run has non-finite values; your search found none"
assert first_nonfinite.endswith("mlp.act_fn"), (
    f"first non-finite module is {first_nonfinite}. Did you walk the record in hook order?")
assert f"layers.{FAULT_LAYER}." in first_nonfinite
assert idx_nan is not None and idx_nan >= idx_nonfinite, (
    "the first NaN cannot come before the first non-finite value")
assert first_divergent == first_nonfinite, (
    f"first divergence ({first_divergent}) should be the module that introduced the fault")
assert anomaly_raised is False, (
    "detect_anomaly only checks backward; a forward-only run with a NaN should complete")
print(f"[OK] fault entered at {first_nonfinite} (record position {idx_nonfinite})")
print(f"     first NaN appears {idx_nan - idx_nonfinite} module(s) later, at {first_nan}")
print("     detect_anomaly stayed silent: the fault is in the forward pass")

In [ ]:
# 1.6 (provided): restore the model, keep what the export needs, free GPU memory.
qwen.model.layers[FAULT_LAYER].mlp.act_fn = _orig_act
part1 = {
    "model": MODEL_SOURCE,
    "n_layers": int(N_LAYERS),
    "n_modules_recorded": int(len(faulty)),
    "fault_layer_index": int(FAULT_LAYER),
    "first_nonfinite": first_nonfinite,
    "idx_nonfinite": int(idx_nonfinite),
    "first_nan": first_nan,
    "idx_nan": int(idx_nan),
    "first_divergent": first_divergent,
    "idx_divergent": int(idx_divergent),
    "baseline_nonfinite_count": int(baseline_nonfinite_count),
    "anomaly_raised": bool(anomaly_raised),
}
del qwen, baseline, faulty, base_logits, faulty_logits
if DEV == "cuda":
    torch.cuda.empty_cache()
print("Part 1 saved; model released.")

---
## 2: ★ TODO: A Recompile Storm

**Scenario.** A serving loop compiles one transformer block with
`dynamic=False` and feeds it prompts of many lengths. Every new length is a
new shape, so every call compiles again. This is the Module 6 "missed bucket"
problem in miniature, and it is a layer-2 (graph capture) failure: each
recompile is a guard failure.

You will **count** the storm three ways (the Dynamo counter, the recompile log,
and the compile IDs in the `TORCH_TRACE` log), watch Dynamo **hit the recompile
limit** and quietly fall back to eager, then **fix** it with shape buckets and
**guard** the fix so any later compile is an error.

TRACE steps: **A** (audit the capture contract) and **C** (correlate compile
events with slow calls).

In [ ]:
# 2.0 (provided): the block used in Parts 2-4 (explicit Q/K/V, the Lab 2 pattern).
class TinyBlock(nn.Module):
    # Pre-norm causal transformer block. Causal mask uses -1e4, not -inf (lecture Module 4, physical layout).
    def __init__(self, d=256, heads=4, ffn=1024):
        super().__init__()
        self.h, self.dh = heads, d // heads
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.up = nn.Linear(d, ffn)
        self.down = nn.Linear(ffn, d)

    def forward(self, x):
        B, S, D = x.shape
        h = self.ln1(x)
        q = self.q(h).view(B, S, self.h, self.dh).transpose(1, 2)
        k = self.k(h).view(B, S, self.h, self.dh).transpose(1, 2)
        v = self.v(h).view(B, S, self.h, self.dh).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        causal = torch.ones(S, S, dtype=torch.bool, device=x.device).triu(1)
        scores = scores.masked_fill(causal, -1e4)
        out = (torch.softmax(scores, dim=-1) @ v).transpose(1, 2).reshape(B, S, D)
        x = x + self.o(out)
        return x + self.down(F.gelu(self.up(self.ln2(x))))


torch.manual_seed(0)
block = TinyBlock().to(DEV).eval()
LENGTHS = [40, 52, 64, 77, 90, 101, 118, 128, 150, 173, 200, 233]
inputs_by_len = {S: torch.randn(1, S, 256, device=DEV) for S in LENGTHS}
print(f"{len(LENGTHS)} distinct prompt lengths; recompile limit = {RECOMPILE_LIMIT}")

In [ ]:
# 2.1 (provided): run the storm with the recompile log on.
dynamo.reset()
counters.clear()
torch._logging.set_logs(recompiles=True)
storm_block = torch.compile(block, dynamic=False)
storm_ms = []
with torch.no_grad():
    for S in LENGTHS:
        sync()
        t0 = time.perf_counter()
        storm_block(inputs_by_len[S])
        sync()
        storm_ms.append((time.perf_counter() - t0) * 1e3)
torch._logging.set_logs()          # back to defaults
print("per-call ms:", [round(t, 1) for t in storm_ms])

In [ ]:
# TODO 2.2: Measure the storm.
#
#   storm_compiles        : int, graphs Dynamo compiled in 2.1
#                           (counters["stats"]["unique_graphs"])
#   storm_eager_calls     : int, calls that did NOT get a compiled graph because
#                           the recompile limit was reached (len(LENGTHS) - storm_compiles,
#                           floored at 0)
#   storm_slowest_ms      : float, max of storm_ms
#   storm_median_ms       : float, median of storm_ms (use torch.tensor(...).median())

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# TODO 2.3: Read the compile IDs from the TORCH_TRACE log (lecture Module 3, compile logs).
#
# Each line of a trace log in TRACE_DIR is a log prefix followed by one JSON
# record. Lines that start with a tab are payload; skip them. Records written
# while Dynamo compiles carry "frame_id" and "frame_compile_id"; the record
# that opens each compile also has a "dynamo_start" key.
#
# Collect the distinct (frame_id, frame_compile_id) pairs from "dynamo_start"
# records in every *.log file, then set
#   trace_compile_ids : sorted list of strings "frame/compile", e.g. "0/3"
#   n_trace_compiles  : int, len(trace_compile_ids), or None if no trace
#                       files exist (TORCH_TRACE was set too late)

trace_files = sorted(glob.glob(os.path.join(TRACE_DIR, "*.log")))
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# 2.4 (provided): the same trace as an HTML report.
TL_OUT = os.path.abspath("lab5_tlparse")
shutil.rmtree(TL_OUT, ignore_errors=True)
try:
    # tlparse reads one log file: pass the newest one in the trace folder.
    _logs = sorted(glob.glob(os.path.join(TRACE_DIR, "*.log")), key=os.path.getmtime)
    if not _logs:
        raise FileNotFoundError("no trace log in " + TRACE_DIR)
    _r = subprocess.run(["tlparse", _logs[-1], "-o", TL_OUT, "--overwrite", "--no-browser"],
                        capture_output=True, text=True, timeout=300)
    print((_r.stdout or _r.stderr)[-800:])
except (FileNotFoundError, subprocess.TimeoutExpired) as e:
    print(f"[!] tlparse did not run ({type(e).__name__}). The compile IDs above are the same data.")
_index = os.path.join(TL_OUT, "index.html")
print("tlparse report:", _index if os.path.exists(_index) else "not produced")
print("Open it from the file browser (left sidebar). Each [frame/compile] entry is one compile.")

In [ ]:
# TODO 2.5: Fix the storm with shape buckets (lecture Module 6, missed buckets).
#
# Write pad_to_bucket(x, buckets) for x of shape [B, S, D]:
#   * pick the smallest bucket b >= S
#   * zero-pad the SEQUENCE dimension at the END up to b (F.pad with (0, 0, 0, b - S))
#   * return (padded_x, S)
#   * if no bucket covers S, raise ValueError -- that is a coverage bug, not
#     something to hide
# Right-padding is safe here: the mask is causal, so real positions never
# attend to pad positions, and you slice the output back to [:, :S].

BUCKETS = (64, 128, 256)


def pad_to_bucket(x, buckets=BUCKETS):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# 2.6 (provided): rerun the same prompts through buckets and check correctness.
dynamo.reset()
counters.clear()
bucket_block = torch.compile(block, dynamic=False)
bucket_ms, bucket_max_rel_err, _pad_tokens = [], 0.0, 0
with torch.no_grad():
    for S in LENGTHS:
        x = inputs_by_len[S]
        xp, n = pad_to_bucket(x)
        _pad_tokens += xp.shape[1] - n
        sync()
        t0 = time.perf_counter()
        y = bucket_block(xp)[:, :n]
        sync()
        bucket_ms.append((time.perf_counter() - t0) * 1e3)
        ref = block(x)
        err = ((y - ref).abs().max() / (ref.abs().max() + 1e-6)).item()
        bucket_max_rel_err = max(bucket_max_rel_err, err)
bucket_compiles = int(counters["stats"]["unique_graphs"])
buckets_used = len({pad_to_bucket(inputs_by_len[S])[0].shape[1] for S in LENGTHS})
pad_overhead = _pad_tokens / sum(LENGTHS)
print(f"bucketed compiles: {bucket_compiles} (buckets used: {buckets_used})")
print(f"max relative error vs eager, unpadded: {bucket_max_rel_err:.2e}")
print(f"padding overhead: {pad_overhead:.1%} extra tokens computed")

In [ ]:
# TODO 2.7: Guard the warm set (lecture Module 6: "fail CI if a kernel compiles after warmup").
#
# Turn on dynamo.config.error_on_recompile, then call bucket_block on an
# UNPADDED input of length 77 (a shape that was never compiled).
# Catch the exception and set
#   guard_fired : bool, True if the call raised
#   guard_error : str, the exception's type name (type(e).__name__)
# Always restore dynamo.config.error_on_recompile = False (try/finally).

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# 2.8 (provided): the timeline of both runs.
fig, ax = plt.subplots(figsize=(9, 3.2), facecolor=CREAM)
_x = range(len(LENGTHS))
ax.bar([i - 0.2 for i in _x], storm_ms, width=0.4, color=TERRACOTTA, label="dynamic=False, no buckets")
ax.bar([i + 0.2 for i in _x], bucket_ms, width=0.4, color=SECONDARY, label="3 buckets")
ax.set_yscale("log")
ax.set_xticks(list(_x))
ax.set_xticklabels(LENGTHS)
ax.set_xlabel("prompt length (tokens), in call order")
ax.set_ylabel("call time (ms, log)")
ax.set_facecolor(CREAM)
for s in ax.spines.values():
    s.set_color(BORDER)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
assert storm_compiles >= 3, f"expected a storm of recompiles, got {storm_compiles} compile(s)"
assert storm_compiles <= RECOMPILE_LIMIT + 1, (
    f"{storm_compiles} compiles exceeds the recompile limit ({RECOMPILE_LIMIT})")
if len(LENGTHS) > RECOMPILE_LIMIT:
    assert storm_eager_calls >= 1, "with more lengths than the limit, some calls must have run eagerly"
assert storm_slowest_ms >= storm_median_ms, "the slowest call cannot be faster than the median"
if n_trace_compiles is not None:
    assert n_trace_compiles >= storm_compiles, (
        f"the trace shows {n_trace_compiles} compile IDs but Dynamo counted {storm_compiles}. "
        f"Restart the runtime and Run All if you re-ran Part 2.")
else:
    print("[!] no TORCH_TRACE files: TORCH_TRACE was set after torch was imported.")
assert bucket_compiles <= buckets_used, (
    f"{bucket_compiles} compiles for {buckets_used} buckets: some shape escaped the buckets")
assert bucket_max_rel_err < 1e-3, f"bucketed output differs from eager by {bucket_max_rel_err:.2e}"
assert 0.0 < pad_overhead < 1.0
assert guard_fired is True, "error_on_recompile should make a new shape raise"
print(f"[OK] storm: {storm_compiles} compiles, then {storm_eager_calls} eager call(s) past the limit")
print(f"     buckets: {bucket_compiles} compiles for {len(LENGTHS)} lengths, "
      f"{pad_overhead:.0%} padding, guard raised {guard_error}")

---
## 3: ★ TODO: A Silent Divergence

**Scenario.** After a compiler upgrade, a model's outputs move slightly. A
loose `allclose` still passes, so nobody notices. One of Inductor's
**decompositions** has been replaced with an approximation. The cell below
plants that bug for you (in a context manager, so it never leaks).

You will first decide whether the difference is a **bug or rounding**, using
the **FP64 criterion** from Module 4. Then you will name the **layer** with
the **four-backend ablation** from Module 3.

TRACE steps: **A** (audit each stage's contract) and **E** (one backend added per experiment).

In [ ]:
# 3.0 (provided): the planted bug, and a clean FP64 reference.
from torch._inductor import decomposition as _ind_decomp

aten = torch.ops.aten


def _tanh_gelu(a, approximate="none"):
    return 0.5 * a * (1 + torch.tanh(math.sqrt(2 / math.pi) * (a + 0.044715 * a * a * a)))


@contextlib.contextmanager
def planted_decomposition_bug():
    key = aten.gelu.default
    had, old = key in _ind_decomp.decompositions, _ind_decomp.decompositions.get(key)
    _ind_decomp.decompositions[key] = _tanh_gelu
    # select_decomp_table() hands Inductor a CACHED COPY of this table
    # (fast_random_decomps is functools.cache'd), so clear it on entry and exit.
    _clear = getattr(_ind_decomp.fast_random_decomps, "cache_clear", lambda: None)
    _clear()
    try:
        yield
    finally:
        if had:
            _ind_decomp.decompositions[key] = old
        else:
            del _ind_decomp.decompositions[key]
        _clear()


# Cached compiles would hide (or keep) the planted bug. Turn caches off for Part 3.
torch._inductor.config.force_disable_caches = True

torch.manual_seed(1)
blk3 = TinyBlock().to(DEV).eval()
x3 = torch.randn(4, 128, 256, device=DEV)
with torch.no_grad():
    eager3 = blk3(x3)
    ref64 = copy.deepcopy(blk3).double()(x3.double())
print(f"eager vs FP64, max abs: {(eager3.double() - ref64).abs().max().item():.2e}")

In [ ]:
# TODO 3.1: Write the FP64 criterion (lecture Module 4, FP64 reference).
#
#   rmse(a, b)  -> float: root-mean-square of (a - b), computed in float64
#   fp64_pass(target, eager, ref, k=2.0, atol=1e-6) -> bool:
#       rmse(target, ref) <= k * rmse(eager, ref) + atol
#
# This is the test torch._dynamo.utils.same applies when given an fp64 reference.

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# 3.2 (provided): a CONTROL run. Clean Inductor, no planted bug.
dynamo.reset()
with torch.no_grad():
    clean_out = torch.compile(blk3, backend="inductor")(x3)
eager_rmse = rmse(eager3, ref64)
clean_inductor_rmse = rmse(clean_out, ref64)
clean_inductor_pass = fp64_pass(clean_out, eager3, ref64)
clean_vs_eager_max = (clean_out - eager3).abs().max().item()
print(f"eager    RMSE vs FP64: {eager_rmse:.3e}")
print(f"inductor RMSE vs FP64: {clean_inductor_rmse:.3e} -> passes: {clean_inductor_pass}")
print(f"inductor vs eager, max abs: {clean_vs_eager_max:.2e}  (a difference, but not a bug)")

In [ ]:
# TODO 3.3: Run the four-backend ablation WITH the planted bug (lecture Module 3, backend ablation).
#
# Inside `with planted_decomposition_bug():`, for each backend in BACKENDS:
#   dynamo.reset(); compile blk3 with that backend; run x3 under torch.no_grad()
# and store
#   ablation[b] = {"rmse_vs_fp64": float, "passes": bool (fp64_pass),
#                  "allclose_loose": bool, torch.allclose(out, eager3, rtol=1e-2, atol=1e-3)}
# Then set
#   first_failing_backend : the first backend in BACKENDS whose "passes" is False
#   divergence_layer      : its layer number from LAYER_OF

BACKENDS = ["eager", "aot_eager", "aot_eager_decomp_partition", "inductor"]
LAYER_OF = {"eager": 2, "aot_eager": 3, "aot_eager_decomp_partition": 4, "inductor": 5}
ablation = {}
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
print(f"{'backend':28s} {'RMSE vs FP64':>14s}  FP64 test  loose allclose")
for b in BACKENDS:
    r = ablation[b]
    print(f"{b:28s} {r['rmse_vs_fp64']:14.3e}  {'pass' if r['passes'] else 'FAIL':9s}  {r['allclose_loose']}")
print(f"\nfirst failing backend: {first_failing_backend} -> layer {divergence_layer}")
torch._inductor.config.force_disable_caches = False

In [ ]:
# --- Checkpoint 3 -------------------------------------------------------
assert clean_inductor_pass is True, (
    "clean Inductor should pass the FP64 criterion: reordering is not a bug. "
    "Check rmse() computes in float64 and k=2, atol=1e-6.")
assert ablation["eager"]["passes"] and ablation["aot_eager"]["passes"], (
    "eager and aot_eager do not use Inductor's decomposition table, so they should pass")
assert not ablation["inductor"]["passes"], "the planted decomposition should fail the FP64 criterion"
assert ablation["inductor"]["allclose_loose"] is True, (
    "the planted bug is meant to be SILENT: a loose allclose should still pass")
assert first_failing_backend == "aot_eager_decomp_partition", (
    f"expected the decomposition stage to fail first, got {first_failing_backend}")
assert divergence_layer == 4
print("[OK] loose allclose passed, the FP64 criterion failed, and the ablation named layer 4")
print(f"     planted RMSE {ablation['inductor']['rmse_vs_fp64']:.2e} vs clean {clean_inductor_rmse:.2e}")

---
## 4: ★ TODO: A Silent CPU Fallback

**Scenario.** A dataflow backend has no kernel for one op. Like torch-spyre, it
runs that op on the host and emits a `FallbackWarning`. The answer stays
correct. The run gets slower, and the warning scrolls past.

Below is a **toy dataflow backend**, written to `toy_dataflow.py` so the
optional probe at the end can import it. It is a real `torch.compile` backend:
AOTAutograd hands it the ATen graph, and it rewrites every unsupported op into a
device→host→device round trip. Its warning text matches torch-spyre's format.

You will **count** the warnings, **promote** the warning to an error to find
the op and its source line, and **see** the round trips in a profiler trace.

TRACE steps: **C** (correlate on a timeline) and **A** (make the contract
violation raise).

In [ ]:
# 4.0 (provided): the toy backend. Ops in UNSUPPORTED run on the host.
TOY_SRC = '''
# A toy dataflow backend for Lab 5. Ops in UNSUPPORTED run on the host.
import os
import warnings
import torch
from torch._dynamo.backends.common import aot_autograd
from functorch.compile import make_boxed_func


class FallbackWarning(UserWarning):
    pass


aten = torch.ops.aten
UNSUPPORTED = set() if os.environ.get("TOY_SUPPORTS_GELU") == "1" else {aten.gelu}
STATS = {"fallback_calls": 0}


def _cpu_fallback(op, where):
    def run(*args, **kwargs):
        dev = next((a.device for a in args if isinstance(a, torch.Tensor)), torch.device("cpu"))
        warnings.warn(f"{op} is falling back to cpu ({where})", FallbackWarning, stacklevel=2)
        STATS["fallback_calls"] += 1
        host = [a.to("cpu") if isinstance(a, torch.Tensor) else a for a in args]
        return op(*host, **kwargs).to(dev)
    return run


def _compile(gm, example_inputs):
    for node in gm.graph.nodes:
        packet = getattr(node.target, "overloadpacket", None)
        if node.op == "call_function" and packet in UNSUPPORTED:
            st = [s.strip() for s in (node.meta.get("stack_trace") or "").splitlines() if s.strip()]
            where = " | ".join(st[-2:]) if st else "source line unknown"
            node.target = _cpu_fallback(node.target, where)
    gm.recompile()
    return make_boxed_func(gm.forward)


toy_dataflow = aot_autograd(fw_compiler=_compile)
'''
with open("toy_dataflow.py", "w") as _f:
    _f.write(TOY_SRC)
import importlib
import toy_dataflow as toy
toy = importlib.reload(toy)
FallbackWarning = toy.FallbackWarning

torch.manual_seed(2)
blk4 = TinyBlock().to(DEV).eval()
x4 = torch.randn(2, 128, 256, device=DEV)
dynamo.reset()
toy_block = torch.compile(blk4, backend=toy.toy_dataflow)
with torch.no_grad():
    eager4 = blk4(x4)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        toy_out = toy_block(x4)          # compile once, quietly
fallback_correct = bool(torch.allclose(toy_out, eager4, rtol=1e-4, atol=1e-5))
print(f"toy backend output matches eager: {fallback_correct}")

In [ ]:
# TODO 4.1: How visible is the fallback?
#
# a) Run toy_block(x4) 5 times inside
#        warnings.catch_warnings(record=True) as w  +  warnings.simplefilter("always")
#    n_fallback_warnings = number of records in w whose category is FallbackWarning
# b) Run it 5 more times with warnings.simplefilter("default") instead
#    (Python's normal behaviour: show a warning once per source location)
#    n_default_shown = number of FallbackWarning records you got this time
# c) fallback_message = str(...) of the first FallbackWarning record's message
#    fallback_op      = fallback_message.split(" is falling back")[0]
# Run everything under torch.no_grad().

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# TODO 4.2: Promote the warning to an error (lecture Module 4, fallback as an error).
#
# Inside warnings.catch_warnings(), call
#     warnings.filterwarnings("error", category=FallbackWarning)
# then run toy_block(x4) under torch.no_grad() in a try/except.
#   promoted_raised  : True if a FallbackWarning was raised as an exception
#   promoted_message : str(e)
# Note: the class filter must be set in code, after the import.
# `python -W error::toy_dataflow.FallbackWarning` would be ignored for a
# package installed in site-packages (CPython issue 66733).

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# TODO 4.3: See the round trips on a timeline (lecture Module 3, profiler timeline, TRACE step C).
#
# Write count_host_copies(fn, x) that profiles ONE call of fn(x) under
# torch.no_grad() with profile(activities=ACTS), calls sync() inside the
# profiled block, and returns (n, prof) where n is the number of profiler events
# named "aten::_to_copy" (one per device<->host move; prof.events()).
# Then measure:
#   n_copies_fallback : for toy_block
#   n_copies_eager    : for blk4 (plain eager, no fallback)
# and export the fallback trace with prof.export_chrome_trace(FALLBACK_TRACE).
# Run the toy block with warnings ignored, so printing does not distort timing.

ACTS = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if DEV == "cuda" else [])
FALLBACK_TRACE = "lab5_fallback_trace.json"
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# --- Checkpoint 4 -------------------------------------------------------
assert fallback_correct is True, "a CPU fallback should still give the right answer here"
assert n_fallback_warnings == 5, f"one gelu per forward, 5 forwards: expected 5, got {n_fallback_warnings}"
assert n_default_shown < n_fallback_warnings, (
    "Python's default filter should show the repeated warning fewer times")
assert fallback_op.startswith("aten.gelu"), f"expected aten.gelu, got {fallback_op!r}"
assert promoted_raised is True, "the promoted FallbackWarning should raise"
if DEV == "cuda":
    assert n_copies_fallback >= n_copies_eager + 2, (
        "a fallback moves data to the host and back: expect at least 2 extra copies")
else:
    print("[i] CPU mode: .to('cpu') is a no-op, so the copy count is not checked")
print(f"[OK] {fallback_op} fell back on every forward; Python showed it {n_default_shown} time(s)")
print(f"     promoted to an error; {n_copies_fallback - n_copies_eager} extra host copies per step")

---
## 5: Update the Accelerator Landscape Map

Week 5's section of the map is a **failure-mode table**: one row per failure,
one column per architecture, recording what the signal looks like there. Your
GPU column is measured. The Spyre and TPU columns are your predictions from
the lecture. Use the layer numbers from the eight-layer model in Module 2.

In [ ]:
# 5.1: One row per planted failure. Fill every None.
#   layer       : int, the debugging layer where the fault ENTERED (1-8)
#   gpu_signal  : str, what you observed on the GPU (use your numbers)
#   spyre_signal: str, what you predict on Spyre / torch-spyre
#   tpu_signal  : str, what you predict on TPU / XLA

landscape_rows_w5 = [
    {"failure": "injected NaN", "layer": None,
     "gpu_signal": None, "spyre_signal": None, "tpu_signal": None},
    {"failure": "recompile storm", "layer": None,
     "gpu_signal": None, "spyre_signal": None, "tpu_signal": None},
    {"failure": "decomposition bug", "layer": None,
     "gpu_signal": None, "spyre_signal": None, "tpu_signal": None},
    {"failure": "CPU fallback", "layer": None,
     "gpu_signal": None, "spyre_signal": None, "tpu_signal": None},
]

In [ ]:
# --- Checkpoint 5 -------------------------------------------------------
assert len(landscape_rows_w5) == 4
for _row in landscape_rows_w5:
    assert all(v is not None and str(v).strip() for v in _row.values()), f"fill every field: {_row}"
    assert isinstance(_row["layer"], int) and 1 <= _row["layer"] <= 8
_layers = [r["layer"] for r in landscape_rows_w5]
assert _layers[1] == 2, "a recompile storm is a graph-capture (guard) failure"
assert _layers[2] == divergence_layer, "the decomposition bug's layer should match your ablation"
assert _layers[3] == 4, "an op with no device kernel is an op-mapping failure"
for _r in landscape_rows_w5:
    print(f"  L{_r['layer']}  {_r['failure']:18s} GPU: {_r['gpu_signal'][:70]}")
print("[OK] Week 5 Landscape Map rows complete")

---

## Check your understanding (ungraded)

These prompts are **not graded and not submitted**. They are the reasoning the
**Concept Check quiz** tests, and that quiz is worth **10 graded points**.
Answer in your own words, citing the numbers *you* measured.

### 1. Where did the NaN come from?
Your first non-finite module and your first NaN module were different. Explain
why, from the values the planted activation produces for negative inputs.
Which debugging layer did the fault *enter* at, and where did the *symptom*
first appear? Why did `detect_anomaly` stay silent?

### 2. Why did the storm stop?
Your Dynamo counter stopped below the number of lengths. What did Dynamo do
for the remaining calls, and why is that dangerous in a serving system? Your
bucket fix computed some padding. Which is worse on Spyre: more buckets or
more padding? Use what Week 4 said about compiled shapes.

### 3. Bug or rounding?
Clean Inductor differed from eager and passed. The planted run passed a loose
`allclose` and failed. State the rule you would put in a CI test, and say why
an FP64 reference is better than comparing to eager alone. Why did `aot_eager`
pass while `aot_eager_decomp_partition` failed?

### 4. A fallback on a dataflow target.
Your fallback was correct and slower. On Spyre, what crosses PCIe, and what
happens to the bundle? Give one way the round trip can change the *numbers*,
not only the speed (lecture Module 4, fallback as an error).

### Comparative callout
For each of the four failures, name the architecture among GPU, TPU, Cerebras,
Groq and Spyre where it would be **most expensive**, and why. Use the Week 2
axis of scheduling commitment.

---
## 6: Save and export your results

The cell below writes **`lab5_results.json`**. It carries the **raw inputs**
next to every derived number, so the grader can recompute each one.

In [ ]:
# === Part 6: Save your results for self-assessment =====================
# PROVIDED - do not edit.
import json as _json

RESULTS = {
    "part1_nan": part1,
    "part2_recompile": {
        "lengths": LENGTHS,
        "recompile_limit": RECOMPILE_LIMIT,
        "storm_compiles": int(storm_compiles),
        "storm_eager_calls": int(storm_eager_calls),
        "storm_ms": [float(t) for t in storm_ms],
        "storm_slowest_ms": float(storm_slowest_ms),
        "storm_median_ms": float(storm_median_ms),
        "trace_compile_ids": trace_compile_ids,
        "n_trace_compiles": n_trace_compiles,
        "buckets": list(BUCKETS),
        "buckets_used": int(buckets_used),
        "bucket_compiles": int(bucket_compiles),
        "bucket_ms": [float(t) for t in bucket_ms],
        "bucket_max_rel_err": float(bucket_max_rel_err),
        "pad_overhead": float(pad_overhead),
        "guard_fired": bool(guard_fired),
        "guard_error": guard_error,
    },
    "part3_divergence": {
        "eager_rmse": float(eager_rmse),
        "clean_inductor_rmse": float(clean_inductor_rmse),
        "clean_inductor_pass": bool(clean_inductor_pass),
        "ablation": ablation,
        "k": 2.0,
        "atol": 1e-6,
        "first_failing_backend": first_failing_backend,
        "divergence_layer": divergence_layer,
    },
    "part4_fallback": {
        "fallback_correct": bool(fallback_correct),
        "n_fallback_warnings": int(n_fallback_warnings),
        "n_default_shown": int(n_default_shown),
        "fallback_op": fallback_op,
        "promoted_raised": bool(promoted_raised),
        "n_copies_fallback": int(n_copies_fallback),
        "n_copies_eager": int(n_copies_eager),
    },
    "landscape_rows_w5": landscape_rows_w5,
    "environment": dict(VERSIONS, mode=MODE),
}

with open("lab5_results.json", "w") as _f:
    _json.dump(RESULTS, _f, indent=2)

print("Saved lab5_results.json\n")
print(_json.dumps(RESULTS, indent=2))
print("\nNEXT STEPS")
print("1. Copy the JSON above, or download lab5_results.json from the file browser.")
print("2. Paste it into the RESULTS_JSON cell of the grader notebook on Coursera Labs and run every cell.")
print("3. Take the Concept Check quiz (10 graded points, 80% to pass).")

---
## ★ Optional: Make the limitation executable (no Spyre needed)

Module 6 ended with a discipline: write each backend gap as a **strict-xfail
probe**. While the gap exists, the probe fails and is marked *expected*. When
the backend fixes the gap, the probe passes, and strict mode turns that
*unexpected pass* into a failure, which tells you to delete the workaround.

The cell below writes a probe for the toy backend's missing `gelu` kernel and
runs it twice: as shipped, and with `TOY_SUPPORTS_GELU=1` (the "fixed" backend).

In [ ]:
PROBE = '''
import warnings
import pytest
import torch
import torch.nn.functional as F
import toy_dataflow as toy


@pytest.mark.xfail(strict=True, reason="toy backend has no gelu kernel (Lab 5, Part 4)")
def test_gelu_runs_on_device():
    f = torch.compile(lambda x: F.gelu(x), backend=toy.toy_dataflow)
    x = torch.randn(64, 64)
    with warnings.catch_warnings():
        warnings.simplefilter("error", toy.FallbackWarning)
        y = f(x)
    torch.testing.assert_close(y, F.gelu(x))
'''
with open("test_toy_probe.py", "w") as _f:
    _f.write(PROBE)

for _label, _env in [("backend as shipped", {}), ("backend 'fixed'", {"TOY_SUPPORTS_GELU": "1"})]:
    # Drop TORCH_TRACE for the child process (an empty value would break its compiles).
    _base = {k: v for k, v in os.environ.items() if k != "TORCH_TRACE"}
    _r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "test_toy_probe.py"],
                        capture_output=True, text=True, env={**_base, **_env})
    _summary = [l for l in _r.stdout.splitlines() if l.strip()][-1:]
    print(f"{_label:20s} exit={_r.returncode}  {_summary}")
print("\nExpected: 1 xfailed (exit 0), then a strict XPASS failure (exit 1). Delete the workaround.")

---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track._

Repeat **Part 4** on torch-spyre, the stack the lecture uses (SNAPSHOT: torch-spyre commit 5a6dc7a; check the repo's debugging guide for changes).

1. Import `torch_spyre`, which registers the `spyre` device, and compile `TinyBlock` with the default `torch.compile` backend.
2. Promote `torch_spyre.ops.fallbacks.FallbackWarning` to an error **in code**. Which op, if any, falls back?
3. Turn on backend logging with `TORCH_LOGS="+torch_spyre.inductor"`, and dump artifacts with `TORCH_COMPILE_DEBUG=1` (look for one `sdsc_*.json` per kernel).
4. Capture one kernel with the OpSpec Lab and replay it: `python tests/op_specs/capture.py repro.py --out captured/`. Is the replay correct? Which side of the frontend/backend split does that clear?
5. Compare the FP64 criterion from Part 3 with the Spyre transfer floor from Module 4 (about 4.9e-4 for inputs in [0, 1)).

In [ ]:
# Spyre track only: skeleton
# import warnings
# import torch_spyre                                   # registers the "spyre" device
# from torch_spyre.ops.fallbacks import FallbackWarning
# warnings.filterwarnings("error", category=FallbackWarning)
# blk_s = TinyBlock().to("spyre").eval()
# x_s = torch.randn(2, 128, 256).to("spyre")
# with torch.no_grad():
#     y_s = torch.compile(blk_s)(x_s)
# ref = blk4.cpu()(x_s.cpu())                          # CPU baseline, same inputs
# print(rmse(y_s.cpu(), ref.double()))

---

### Wrap-up checklist
- [ ] Preflight ran **first**, in a fresh runtime (so `TORCH_TRACE` took effect).
- [ ] Part 1: the first non-finite module, the first NaN and the first divergence are found, and `detect_anomaly` reported nothing.
- [ ] Part 2: the recompilations are counted three ways, the buckets reduce them to three, and the recompile guard raised an error.
- [ ] Part 3: the FP64 criterion separates rounding from the bug, and the ablation identifies layer 4.
- [ ] Part 4: the fallback is counted, promoted to an error, and found on the profiler timeline.
- [ ] Part 5: the four Landscape Map rows are filled in.
- [ ] Part 6 ran, and `lab5_results.json` is written and pasted into the grader notebook.
- [ ] The **Concept Check quiz** is complete: 10 graded points, 80% to pass.

_End of Lab 5. Next week, Roofline Analysis measures whether a bottleneck can be removed._

---

*Course: Dataflow Architectures for AI, Week 5 Lab, v2.0 | © 2026 Dr. Kaoutar El Maghraoui*